# Harmonize database metadata

This notebook harmonizes organism names, general tissues, and specific tissues across CELLxGENE, HuBMAP, and the Museum of Spatial Transcriptomics. The mapping is stored explicitly in `metadata_harmonization_mapping.json` so it can be reviewed and edited without digging through notebook code.


In [19]:
from ast import literal_eval
from pathlib import Path
import json
import re

import pandas as pd

DATA_DIR = Path(".")
if not (DATA_DIR / "cellxgene_meta.csv").exists():
    DATA_DIR = Path("data_availability_quantification")

MAPPING_PATH = DATA_DIR / "metadata_harmonization_mapping.json"
OUTPUT_PATH = DATA_DIR / "harmonized_metadata.csv"
CXG_OUTPUT_PATH = DATA_DIR / "cellxgene_meta_harmonized.csv"
HUBMAP_OUTPUT_PATH = DATA_DIR / "hubmap_meta_harmonized.csv"
MOST_OUTPUT_PATH = DATA_DIR / "museum_of_st_meta_harmonized.csv"


In [20]:
most_df = pd.read_csv(DATA_DIR / "museum_of_st_meta.csv")
cxg_df = pd.read_csv(DATA_DIR / "cellxgene_meta.csv")
hubmap_df = pd.read_csv(DATA_DIR / "hubmap_meta.csv")

with MAPPING_PATH.open() as f:
    harmonization_mapping = json.load(f)

print(f"CELLxGENE rows: {len(cxg_df):,}")
print(f"HuBMAP rows: {len(hubmap_df):,}")
print(f"Museum of ST rows: {len(most_df):,}")
print(f"Mapping file: {MAPPING_PATH}")


CELLxGENE rows: 645
HuBMAP rows: 4,704
Museum of ST rows: 1,072
Mapping file: metadata_harmonization_mapping.json


In [21]:
LIST_COLUMNS = ["organisms", "tissue_general", "tissues"]


def parse_list_value(value):
    """Parse list-like metadata values while tolerating empty strings and scalars."""
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return []

    try:
        parsed = literal_eval(text)
    except (ValueError, SyntaxError):
        return [text]

    if isinstance(parsed, list):
        return parsed
    if isinstance(parsed, tuple):
        return list(parsed)
    if pd.isna(parsed):
        return []
    return [parsed]


def normalize_text(value):
    text = str(value).strip().replace("_", " ")
    text = text.replace("×", "x")
    text = re.sub(r"\s+", " ", text)
    return text.casefold()


def canonicalize(value, mapping_section):
    normalized = normalize_text(value)
    aliases = harmonization_mapping.get(mapping_section, {})
    if normalized in aliases:
        return aliases[normalized]

    if mapping_section == "organisms":
        return normalized.replace(" ", "_")

    if mapping_section == "tissue_general":
        for rule in harmonization_mapping.get("tissue_general_keyword_rules", []):
            if any(normalize_text(pattern) in normalized for pattern in rule["contains"]):
                return rule["canonical"]

    return normalized


def unique_preserve_order(values):
    seen = set()
    deduped = []
    for value in values:
        if value in seen:
            continue
        seen.add(value)
        deduped.append(value)
    return deduped


def harmonize_list_column(series, mapping_section):
    return series.apply(
        lambda value: unique_preserve_order(
            canonicalize(item, mapping_section)
            for item in parse_list_value(value)
            if str(item).strip()
        )
    )


def list_to_readable_string(values):
    return "; ".join(values)


In [22]:
def harmonize_metadata(df, source_name):
    harmonized = df.copy()

    for column in LIST_COLUMNS:
        if column not in harmonized.columns:
            harmonized[column] = "[]"

    harmonized["organisms_harmonized"] = harmonize_list_column(harmonized["organisms"], "organisms")
    harmonized["tissue_general_harmonized"] = harmonize_list_column(harmonized["tissue_general"], "tissue_general")
    harmonized["tissues_harmonized"] = harmonize_list_column(harmonized["tissues"], "tissues")

    # Keep human-readable string columns for CSV consumers that do not parse Python lists.
    harmonized["organisms_harmonized_label"] = harmonized["organisms_harmonized"].apply(list_to_readable_string)
    harmonized["tissue_general_harmonized_label"] = harmonized["tissue_general_harmonized"].apply(list_to_readable_string)
    harmonized["tissues_harmonized_label"] = harmonized["tissues_harmonized"].apply(list_to_readable_string)

    harmonized["source_database"] = harmonized.get("source_database", source_name)
    return harmonized


cxg_harmonized = harmonize_metadata(cxg_df, "cellxgene")
hubmap_harmonized = harmonize_metadata(hubmap_df, "hubmap")
most_harmonized = harmonize_metadata(most_df, "museum_of_spatial_transcriptomics")

shared_columns = sorted(
    set(cxg_harmonized.columns)
    .union(hubmap_harmonized.columns)
    .union(most_harmonized.columns)
)
harmonized_df = pd.concat(
    [
        cxg_harmonized.reindex(columns=shared_columns),
        hubmap_harmonized.reindex(columns=shared_columns),
        most_harmonized.reindex(columns=shared_columns),
    ],
    ignore_index=True,
)

cxg_harmonized.to_csv(CXG_OUTPUT_PATH, index=False)
hubmap_harmonized.to_csv(HUBMAP_OUTPUT_PATH, index=False)
most_harmonized.to_csv(MOST_OUTPUT_PATH, index=False)
harmonized_df.to_csv(OUTPUT_PATH, index=False)

# Re-save the loaded mapping with deterministic formatting so edits remain readable.
with MAPPING_PATH.open("w") as f:
    json.dump(harmonization_mapping, f, indent=2, sort_keys=True)
    f.write("\n")

print(f"Wrote {CXG_OUTPUT_PATH}")
print(f"Wrote {HUBMAP_OUTPUT_PATH}")
print(f"Wrote {MOST_OUTPUT_PATH}")
print(f"Wrote {OUTPUT_PATH}")
print(f"Re-saved readable mapping at {MAPPING_PATH}")


Wrote cellxgene_meta_harmonized.csv
Wrote hubmap_meta_harmonized.csv
Wrote museum_of_st_meta_harmonized.csv
Wrote harmonized_metadata.csv
Re-saved readable mapping at metadata_harmonization_mapping.json


In [23]:
def summarize_values(df, columns):
    rows = []
    for column in columns:
        values = df[column].explode()
        values = values[values.notna() & (values != "")]
        rows.append(
            {
                "column": column,
                "n_unique": values.nunique(),
                "top_values": values.value_counts().head(10).to_dict(),
            }
        )
    return pd.DataFrame(rows)


summary = summarize_values(
    harmonized_df,
    ["organisms_harmonized", "tissue_general_harmonized", "tissues_harmonized"],
)
summary


,column,n_unique,top_values
0,organisms_harmonized,51,"{'homo_sapiens': 5879, 'mus_musculus': 381, 'c..."
1,tissue_general_harmonized,457,"{'placenta': 1167, 'lung': 1039, 'kidney': 910..."
2,tissues_harmonized,1289,"{'placenta': 1166, 'lung': 1004, 'kidney': 897..."


In [25]:
preview_columns = [
    "source_database",
    "dataset_id",
    "organisms",
    "organisms_harmonized_label",
    "tissue_general",
    "tissue_general_harmonized_label",
    "tissues",
    "tissues_harmonized_label",
]

harmonized_df[preview_columns].head(20)


,source_database,dataset_id,organisms,organisms_harmonized_label,tissue_general,tissue_general_harmonized_label,tissues,tissues_harmonized_label
0,cellxgene,00476f9f-ebc1-4b72-b541-32f912ce36ea,['homo_sapiens'],homo_sapiens,['brain'],brain,['hypothalamus'],hypothalamus
1,cellxgene,0087cde2-967d-4f7c-8e6e-40e4c9ad1891,['homo_sapiens'],homo_sapiens,['brain'],brain,['neocortex'],neocortex
2,cellxgene,01209dce-3575-4bed-b1df-129f57fbc031,['homo_sapiens'],homo_sapiens,"['blood', 'bone marrow', 'lung', 'lymph node']",blood; bone marrow; lung; lymph node,"['bone marrow', 'bronchopulmonary lymph node',...",bone marrow; bronchopulmonary lymph node; lung...
3,cellxgene,0129dbd9-a7d3-4f6b-96b9-1da155a93748,['homo_sapiens'],homo_sapiens,['eye'],eye,"['fovea centralis', 'macula lutea proper', 'pe...",fovea centralis; macula lutea proper; peripher...
4,cellxgene,019c7af2-c827-4454-9970-44d5e39ce068,['homo_sapiens'],homo_sapiens,"['colon', 'small intestine']",colon; small intestine,"['ascending colon', 'descending colon', 'duode...",ascending colon; descending colon; duodenum; i...
5,cellxgene,01ad3cd7-3929-4654-84c0-6db05bd5fd59,['homo_sapiens'],homo_sapiens,['blood'],blood,['blood'],blood
6,cellxgene,01b84709-139c-4485-98a9-6e14c58fbbf6,['homo_sapiens'],homo_sapiens,['brain'],brain,"['frontal lobe', 'lobe of cerebral hemisphere'...",frontal lobe; lobe of cerebral hemisphere; par...
7,cellxgene,01c93cf6-b695-4e30-a26e-121ae8b16a9e,['homo_sapiens'],homo_sapiens,['blood'],blood,['blood'],blood
8,cellxgene,02792605-4760-4023-82ad-40fc4458a5db,['homo_sapiens'],homo_sapiens,['liver'],liver,['caudate lobe of liver'],caudate lobe of liver
9,cellxgene,03d38670-1444-4001-bc53-9936e61d9b20,['homo_sapiens'],homo_sapiens,['brain'],brain,['hypothalamus'],hypothalamus
